**1. Carga de datos `load_locations.py`**

In [2]:
import redis
from locations import POIS as locations

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

for poi in locations:
    poi_id = poi["id"]
    name = poi["name"]
    lon = poi["lon"]
    lat = poi["lat"]

    r.geoadd("poi:locations", (lon, lat, poi_id))
    r.hset("poi:info", poi_id, name)

print("Datos cargados correctamente.")


Datos cargados correctamente.


**2. Búsqueda por radio `find_by_radius.py`**

In [4]:
import redis
import sys

lat = float(40.41677)
lon = float(-3.70379)

if len(sys.argv) == 4:
    distance = float(sys.argv[3])
else:
    distance = 2000 
    
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

results = r.geosearch(
    "poi:locations",
    longitude=lon,
    latitude=lat,
    radius=distance,
    unit="m"
)

print(f"Encontrados {len(results)} POIs en {distance/1000} km:")

for poi_id in results:
    name = r.hget("poi:info", poi_id)
    print(f"-> {name} ({poi_id})")



Encontrados 17 POIs en 2.0 km:
-> Catedral de la Almudena (poi_012)
-> Palacio Real (poi_004)
-> Templo de Debod (poi_010)
-> Plaza de Cascorro (El Rastro) (poi_018)
-> Mercado de San Miguel (poi_011)
-> Plaza Mayor (poi_005)
-> Puerta del Sol (poi_001)
-> Museo Reina Sofía (poi_006)
-> CaixaForum Madrid (poi_017)
-> Museo del Prado (poi_002)
-> Museo Thyssen-Bornemisza (poi_007)
-> Gran Vía (Plaza Callao) (poi_009)
-> Plaza de España (poi_016)
-> Plaza de Cibeles (poi_014)
-> Parque del Retiro (poi_003)
-> Puerta de Alcalá (poi_015)
-> Estación de Atocha (poi_013)


**3. Búsqueda del “más cercano” `find_nearest.py`**

In [8]:
import redis
import sys

lat = float(input("Introduce tu latitud: "))
lon = float(input("Introduce tu longitud: "))

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

result = r.geosearch(
    "poi:locations",
    longitude=lon,
    latitude=lat,
    radius=50000, 
    unit="m",
    count=1,
    withdist=True
)

if not result:
    print("No se encontró ningún POI.")
else:
    poi_id, distance = result[0]
    name = r.hget("poi:info", poi_id)
    print(f'El POI más cercano es "{name}", que está a {distance/1000:.2f} km.')


El POI más cercano es "Estadio Santiago Bernabéu", que está a 10.50 km.
